In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ModuleNotFoundError:
    print("Not running in Google Colab; skipping Drive mount.")


In [ ]:
from pathlib import Path
import os


def resolve_data_path(filename: str) -> str:
    """
    Resolve data files in local Jupyter or Google Colab.
    In Colab, mount Google Drive first and place data under:
    /content/drive/MyDrive/feature-selection-experiments/data
    """
    candidates = []

    env_data_dir = os.environ.get("FEATURE_SELECTION_DATA_DIR")
    if env_data_dir:
        candidates.append(Path(env_data_dir) / filename)

    cwd = Path.cwd()
    candidates.extend([
        cwd / "data" / filename,
        cwd.parent / "data" / filename,
        Path("/content/data") / filename,
        Path("/content/feature-selection-experiments/data") / filename,
        Path("/content/drive/MyDrive/feature-selection-experiments/data") / filename,
        Path("/content/drive/MyDrive/data") / filename,
        Path(r"D:\feature-selection-experiments\data") / filename,
    ])

    for path in candidates:
        if path.exists():
            return str(path)

    search_roots = [
        Path("/content"),
        Path("/content/drive/MyDrive"),
    ]
    for root in search_roots:
        if not root.exists():
            continue
        matches = list(root.glob(f"**/{filename}"))
        for match in matches:
            if match.parent.name == "data":
                return str(match)
        if matches:
            return str(matches[0])

    checked = "\n".join(str(path) for path in candidates)
    raise FileNotFoundError(
        f"Could not find {filename}. Checked:\n{checked}\n\n"
        "Upload the data folder to /content/data, or mount Google Drive and place it at "
        "/content/drive/MyDrive/feature-selection-experiments/data."
    )


DATA_DIR = Path(resolve_data_path("X_train.csv")).parent
if Path("/content/drive/MyDrive").exists():
    RESULT_DIR = Path("/content/drive/MyDrive/feature-selection-experiments/results/ibabc-cgo")
else:
    RESULT_DIR = Path.cwd() / "results" / "ibabc-cgo"
RESULT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR = RESULT_DIR


def output_path(filename: str) -> str:
    return str(RESULT_DIR / filename)


print(f"Using DATA_DIR: {DATA_DIR}")
print(f"Using RESULT_DIR: {RESULT_DIR}")


In [21]:
import numpy as np
import pandas as pd
import time
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report
from typing import Tuple, List, Dict, Optional

In [22]:
def load_data(
    x_path: str,
    y_path: str
) -> Tuple[pd.DataFrame, np.ndarray]:
    X = pd.read_csv(x_path)
    y_df = pd.read_csv(y_path)

    if y_df.shape[1] != 1:
        raise ValueError("y_train.csv phải chỉ có đúng 1 cột nhãn.")

    y = y_df.iloc[:, 0].values

    # Encode nhãn nếu là chuỗi
    if y.dtype == object or str(y.dtype).startswith("string"):
        le = LabelEncoder()
        y = le.fit_transform(y)

    return X, y

In [23]:
X_path = resolve_data_path("X_train.csv")
y_path = resolve_data_path("y_train.csv")

X, y = load_data(X_path, y_path)

In [24]:
def save_pso_history(history):
    def convert_for_json(obj):
        if isinstance(obj, (int, float, str)):
            return obj
        elif isinstance(obj, (list, tuple)):
            return [convert_for_json(i) for i in obj]
        elif hasattr(obj, 'tolist'):
            return obj.tolist()
        else:
            return str(obj)

    def expand_vector_list(vector_list, prefix):
        arr = np.array([convert_for_json(vec) for vec in vector_list])
        df = pd.DataFrame(arr, columns=[f"{prefix}_{i}" for i in range(arr.shape[1])])
        return df

    df = pd.DataFrame({
        "epoch": list(range(1, len(history["list_global_best_fit"]) + 1)),
        "global_best_fit": history["list_global_best_fit"],
        "current_best_fit": history["list_current_best_fit"],
        "diversity": history["list_diversity"],
        "exploration": history["list_exploration"],
        "exploitation": history["list_exploitation"],
        "exploration_threshold": [0.1] * len(history["list_global_best_fit"]),
        "epoch_time": history["list_epoch_time"]
    })

    df = pd.concat([
        df,
        expand_vector_list(history["list_global_best"], "gb"),
        expand_vector_list(history["list_current_best"], "cb"),
        expand_vector_list(history["list_global_worst"], "gw"),
        expand_vector_list(history["list_current_worst"], "cw"),
    ], axis=1)

    # Save CSV
    df.to_csv(output_path("optimization_metrics_ibabc_cgo_with_formulas.csv"), index=False)
    print("[Metrics] Saved optimization_metrics_ibabc_cgo_with_formulas.csv")

In [25]:


def hamming_distance_ratio(mask_a: np.ndarray, mask_b: np.ndarray) -> float:
    """Return the normalized Hamming distance between two binary masks."""
    mask_a = np.asarray(mask_a).astype(int)
    mask_b = np.asarray(mask_b).astype(int)
    if mask_a.shape != mask_b.shape:
        raise ValueError("Two masks must have the same shape.")
    return float(np.mean(mask_a != mask_b))


def continuous_population_to_masks(
    population: np.ndarray,
    threshold: float = 0.5,
    min_features: int = 1,
    use_sigmoid: bool = False
) -> np.ndarray:
    """Convert a continuous population to binary feature masks for metric tracking."""
    values = 1.0 / (1.0 + np.exp(-population)) if use_sigmoid else np.asarray(population)
    masks = (values > threshold).astype(int)

    if min_features > 0:
        for i in range(masks.shape[0]):
            if masks[i].sum() < min_features:
                top_idx = np.argsort(values[i])[-min_features:]
                masks[i, top_idx] = 1

    return masks


def compute_population_diversity(population_masks: np.ndarray) -> float:
    """Average pairwise Hamming distance among all masks in the current population."""
    population_masks = np.asarray(population_masks).astype(int)
    n_individuals = population_masks.shape[0]
    if n_individuals < 2:
        return 0.0

    total_distance = 0.0
    pair_count = 0
    for i in range(n_individuals):
        for j in range(i + 1, n_individuals):
            total_distance += hamming_distance_ratio(population_masks[i], population_masks[j])
            pair_count += 1

    return float(total_distance / pair_count)


def compute_exploration_exploitation(
    new_masks: np.ndarray,
    previous_masks: np.ndarray,
    similarity_threshold: float = 0.1
) -> Tuple[float, float]:
    """
    Exploration is the fraction of new masks whose nearest previous neighbor
    is farther than the threshold. Exploitation is the complementary fraction.
    """
    new_masks = np.asarray(new_masks).astype(int)
    previous_masks = np.asarray(previous_masks).astype(int)
    if len(new_masks) == 0 or len(previous_masks) == 0:
        return 0.0, 0.0

    exploration_count = 0
    for new_mask in new_masks:
        nearest_distance = min(
            hamming_distance_ratio(new_mask, previous_mask)
            for previous_mask in previous_masks
        )
        if nearest_distance > similarity_threshold:
            exploration_count += 1

    exploration = exploration_count / len(new_masks)
    exploitation = 1.0 - exploration
    return float(exploration), float(exploitation)


def extract_solution_vector(individual):
    """Extract a solution vector from common mealpy Agent/history representations."""
    if hasattr(individual, "solution"):
        return np.asarray(individual.solution, dtype=float)
    if isinstance(individual, dict):
        for key in ("solution", "position", "pos"):
            if key in individual:
                return np.asarray(individual[key], dtype=float)
    if isinstance(individual, (list, tuple)) and len(individual) > 0:
        first = individual[0]
        if np.asarray(first).ndim > 0:
            return np.asarray(first, dtype=float)
    return np.asarray(individual, dtype=float)


def extract_population_matrix(population):
    """Convert a mealpy history population entry to a 2D numpy matrix."""
    return np.asarray([extract_solution_vector(ind) for ind in population], dtype=float)


def compute_formula_metrics_from_history_populations(
    populations,
    threshold: float = 0.5,
    min_features: int = 1,
    similarity_threshold: float = 0.1
) -> Tuple[List[float], List[float], List[float]]:
    """Compute diversity/exploration/exploitation for each epoch from stored populations."""
    diversities = []
    explorations = []
    exploitations = []
    previous_masks = None

    for population in populations:
        population_matrix = extract_population_matrix(population)
        current_masks = continuous_population_to_masks(
            population_matrix,
            threshold=threshold,
            min_features=min_features
        )
        diversities.append(compute_population_diversity(current_masks))

        if previous_masks is None:
            explorations.append(0.0)
            exploitations.append(0.0)
        else:
            exploration, exploitation = compute_exploration_exploitation(
                new_masks=current_masks,
                previous_masks=previous_masks,
                similarity_threshold=similarity_threshold
            )
            explorations.append(exploration)
            exploitations.append(exploitation)

        previous_masks = current_masks.copy()

    return diversities, explorations, exploitations

def transfer_function_S(x):
    return 1 / (1 + np.exp(-x))

def transfer_function_V(x):
    return np.abs(np.tanh(x))

def binary_conversion(x, transfer_function):
    prob = transfer_function(x)
    rand = np.random.rand(*x.shape)
    binary = np.where(rand < prob, 1, 0)
    return binary

def cgo_update(Xi, MG, GB):
    r = np.random.rand()
    if r < 0.25:
        return Xi + np.random.rand() * (np.random.rand() * GB - np.random.rand() * MG)
    elif r < 0.5:
        return GB + np.random.rand() * (np.random.rand() * Xi - np.random.rand() * MG)
    elif r < 0.75:
        return MG + np.random.rand() * (np.random.rand() * Xi - np.random.rand() * GB)
    else:
        k = np.random.randint(0, Xi.shape[0])
        Xi[k] += np.random.uniform(-1, 1)
        return Xi

def fitness_function(solution, X, y, alpha=0.1):
    if np.sum(solution) == 0:
        return 1.0

    if hasattr(X, 'iloc'):
        selected_features = X.iloc[:, solution == 1].values
    else:
        selected_features = X[:, solution == 1]

    if selected_features.shape[1] == 0:
        return 1.0

    # ép y về numpy nếu cần
    if hasattr(y, 'values'):
        y_array = y.values.ravel()
    else:
        y_array = np.ravel(y)

    X_tr, X_val, y_tr, y_val = train_test_split(
        selected_features,
        y_array,
        test_size=0.2,
        random_state=42,
        stratify=y_array
    )

    clf = KNeighborsClassifier(n_neighbors=5)
    clf.fit(X_tr, y_tr)
    acc = clf.score(X_val, y_val)

    error_rate = 1 - acc
    feature_ratio = np.sum(solution) / solution.shape[0]

    return alpha * error_rate + (1 - alpha) * feature_ratio
    
def iBABC_CGO(X_train, y_train,
              num_bees=20, num_islands=4, max_iter=100, limit=10,
              migration_freq=10, migration_rate=0.3, transfer_func='S'):

    if hasattr(X_train, 'values'):
        X_train_np = X_train.values
    else:
        X_train_np = X_train

    if hasattr(y_train, 'values'):
        y_train_np = y_train.values.ravel()
    else:
        y_train_np = y_train.ravel()

    dim = X_train_np.shape[1]
    num_bees_per_island = num_bees // num_islands
    population = np.random.uniform(low=-1, high=1, size=(num_bees, dim))
    fitness = np.zeros(num_bees)
    trial = np.zeros(num_bees)

    tf = transfer_function_S if transfer_func == 'S' else transfer_function_V

    for i in range(num_bees):
        binary_solution = binary_conversion(population[i], tf)
        fitness[i] = fitness_function(binary_solution, X_train_np, y_train_np)

    best_index = np.argmin(fitness)
    global_best = population[best_index].copy()
    global_best_fitness = fitness[best_index]

    total_start_time = time.time()
    epoch_times = []
    no_improve_counter = 0

    history = {
        "list_global_best_fit": [],
        "list_current_best_fit": [],
        "list_global_best": [],
        "list_current_best": [],
        "list_global_worst": [],
        "list_current_worst": [],
        "list_diversity": [],
        "list_exploration": [],
        "list_exploitation": [],
        "list_epoch_time": [],
        "list_population": []
    }

    for iteration in range(max_iter):
        epoch_start = time.time()
        previous_masks_for_metrics = continuous_population_to_masks(
            population,
            threshold=0.5,
            min_features=1,
            use_sigmoid=True
        )

        for i in range(num_bees):
            k = np.random.randint(num_bees)
            while k == i:
                k = np.random.randint(num_bees)
            phi = np.random.uniform(-1, 1, size=dim)
            v = population[i] + phi * (population[i] - population[k])
            binary_v = binary_conversion(v, tf)
            fitness_v = fitness_function(binary_v, X_train_np, y_train_np)
            if fitness_v < fitness[i]:
                population[i] = v
                fitness[i] = fitness_v
                trial[i] = 0
            else:
                trial[i] += 1

        prob = (fitness.max() - fitness) / (fitness.max() - fitness.min() + 1e-9)
        for i in range(num_bees):
            if np.random.rand() < prob[i]:
                k = np.random.randint(num_bees)
                while k == i:
                    k = np.random.randint(num_bees)
                phi = np.random.uniform(-1, 1, size=dim)
                v = population[i] + phi * (population[i] - population[k])
                binary_v = binary_conversion(v, tf)
                fitness_v = fitness_function(binary_v, X_train_np, y_train_np)
                if fitness_v < fitness[i]:
                    population[i] = v
                    fitness[i] = fitness_v
                    trial[i] = 0
                else:
                    trial[i] += 1

        for i in range(num_bees):
            if trial[i] > limit:
                if np.random.rand() < 0.5:
                    population[i] = np.random.uniform(low=-1, high=1, size=dim)
                else:
                    mutation_mask = np.random.rand(dim) < 0.3
                    population[i][mutation_mask] = np.random.uniform(low=-1, high=1, size=np.sum(mutation_mask))
                binary_solution = binary_conversion(population[i], tf)
                fitness[i] = fitness_function(binary_solution, X_train_np, y_train_np)
                trial[i] = 0

        if iteration > 0 and iteration % migration_freq == 0:
            for island_idx in range(num_islands):
                start = island_idx * num_bees_per_island
                end = start + num_bees_per_island
                island = population[start:end]
                island_fitness = fitness[start:end]
                num_migrate = int(migration_rate * num_bees_per_island)
                migrate_indices = np.random.choice(num_bees_per_island, size=num_migrate, replace=False)
                MG = np.mean(island, axis=0)
                for idx in migrate_indices:
                    updated_solution = cgo_update(island[idx], MG, global_best)
                    updated_solution += np.random.normal(0, 0.1, size=dim)
                    binary_updated = binary_conversion(updated_solution, tf)
                    updated_fitness = fitness_function(binary_updated, X_train_np, y_train_np)
                    island[idx] = updated_solution
                    island_fitness[idx] = updated_fitness
                population[start:end] = island
                fitness[start:end] = island_fitness

        best_index = np.argmin(fitness)
        if fitness[best_index] < global_best_fitness:
            global_best = population[best_index].copy()
            global_best_fitness = fitness[best_index]
            no_improve_counter = 0
        else:
            no_improve_counter += 1

        if no_improve_counter >= 10:
            num_reset = int(0.2 * num_bees)
            reset_indices = np.random.choice(num_bees, size=num_reset, replace=False)
            for idx in reset_indices:
                population[idx] = np.random.uniform(low=-1, high=1, size=dim)
                binary_solution = binary_conversion(population[idx], tf)
                fitness[idx] = fitness_function(binary_solution, X_train_np, y_train_np)
                trial[idx] = 0
            no_improve_counter = 0

        epoch_end = time.time()
        epoch_times.append(epoch_end - epoch_start)

        current_best_fit = np.min(fitness)
        current_best_idx = np.argmin(fitness)
        current_worst_fit = np.max(fitness)
        current_worst_idx = np.argmax(fitness)
        current_masks_for_metrics = continuous_population_to_masks(
            population,
            threshold=0.5,
            min_features=1,
            use_sigmoid=True
        )
        diversity = compute_population_diversity(current_masks_for_metrics)
        exploration, exploitation = compute_exploration_exploitation(
            new_masks=current_masks_for_metrics,
            previous_masks=previous_masks_for_metrics,
            similarity_threshold=0.1
        )

        history["list_global_best_fit"].append(global_best_fitness)
        history["list_current_best_fit"].append(current_best_fit)
        history["list_global_best"].append(global_best.copy())
        history["list_current_best"].append(population[current_best_idx].copy())
        history["list_global_worst"].append(population[np.argmax(fitness)].copy())
        history["list_current_worst"].append(population[current_worst_idx].copy())
        history["list_diversity"].append(diversity)
        history["list_exploration"].append(exploration)
        history["list_exploitation"].append(exploitation)
        history["list_epoch_time"].append(epoch_end - epoch_start)
        history["list_population"].append(population.copy())


        print(f"Iteration {iteration:03d} | Best Fitness: {global_best_fitness:.5f} | Diversity: {diversity:.4f} | Exploration: {exploration:.4f} | Exploitation: {exploitation:.4f} | Epoch Time: {epoch_end - epoch_start:.2f}s")

    total_end_time = time.time()
    total_duration = total_end_time - total_start_time
    avg_epoch_time = np.mean(epoch_times)

    print("Training Complete")
    print(f"Total Time: {total_duration:.2f} seconds")
    print(f"Average Epoch Time: {avg_epoch_time:.2f} seconds")

    best_binary = binary_conversion(global_best, tf)
    selected_indices = np.where(best_binary == 1)[0]

    save_pso_history(history)

    return selected_indices

In [26]:
from sklearn.model_selection import train_test_split

selected_indices = iBABC_CGO(X, y)
print(f"Number of selected features: {len(selected_indices)}")

Iteration 000 | Best Fitness: 0.46858 | Epoch Time: 5.79s
Iteration 001 | Best Fitness: 0.46858 | Epoch Time: 5.05s
Iteration 002 | Best Fitness: 0.46858 | Epoch Time: 5.45s
Iteration 003 | Best Fitness: 0.46679 | Epoch Time: 4.93s
Iteration 004 | Best Fitness: 0.45478 | Epoch Time: 4.99s
Iteration 005 | Best Fitness: 0.45135 | Epoch Time: 4.41s
Iteration 006 | Best Fitness: 0.45135 | Epoch Time: 6.41s
Iteration 007 | Best Fitness: 0.45135 | Epoch Time: 5.62s
Iteration 008 | Best Fitness: 0.45135 | Epoch Time: 5.61s
Iteration 009 | Best Fitness: 0.45135 | Epoch Time: 5.31s
Iteration 010 | Best Fitness: 0.45135 | Epoch Time: 6.35s
Iteration 011 | Best Fitness: 0.45135 | Epoch Time: 5.75s
Iteration 012 | Best Fitness: 0.45135 | Epoch Time: 6.02s
Iteration 013 | Best Fitness: 0.45135 | Epoch Time: 5.30s
Iteration 014 | Best Fitness: 0.45135 | Epoch Time: 5.62s
Iteration 015 | Best Fitness: 0.45135 | Epoch Time: 6.79s
Iteration 016 | Best Fitness: 0.45135 | Epoch Time: 5.66s
Iteration 017 

/tmp/ipykernel_106/3705621695.py:2: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-x))


Iteration 056 | Best Fitness: 0.44575 | Epoch Time: 4.99s
Iteration 057 | Best Fitness: 0.44575 | Epoch Time: 5.06s
Iteration 058 | Best Fitness: 0.44575 | Epoch Time: 4.84s
Iteration 059 | Best Fitness: 0.44575 | Epoch Time: 4.95s
Iteration 060 | Best Fitness: 0.44575 | Epoch Time: 7.11s
Iteration 061 | Best Fitness: 0.44575 | Epoch Time: 5.33s
Iteration 062 | Best Fitness: 0.44575 | Epoch Time: 5.21s


/tmp/ipykernel_106/3705621695.py:2: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-x))
/tmp/ipykernel_106/3705621695.py:2: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-x))


Iteration 063 | Best Fitness: 0.44575 | Epoch Time: 4.71s


/tmp/ipykernel_106/3705621695.py:2: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-x))


... (63 outputs left out)

/tmp/ipykernel_106/3705621695.py:2: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-x))
/tmp/ipykernel_106/3705621695.py:2: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-x))
/tmp/ipykernel_106/3705621695.py:2: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-x))
/tmp/ipykernel_106/3705621695.py:2: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-x))
/tmp/ipykernel_106/3705621695.py:2: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-x))
/tmp/ipykernel_106/3705621695.py:2: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-x))
/tmp/ipykernel_106/3705621695.py:2: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-x))
/tmp/ipykernel_106/3705621695.py:2: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-x))
/tmp/ipykernel_106/3705621695.py:2: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-x))
/tmp/ipyke

Iteration 098 | Best Fitness: 0.44575 | Epoch Time: 4.76s


/tmp/ipykernel_106/3705621695.py:2: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-x))
/tmp/ipykernel_106/3705621695.py:2: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-x))
/tmp/ipykernel_106/3705621695.py:2: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-x))
/tmp/ipykernel_106/3705621695.py:2: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-x))
/tmp/ipykernel_106/3705621695.py:2: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-x))
/tmp/ipykernel_106/3705621695.py:2: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-x))
/tmp/ipykernel_106/3705621695.py:2: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-x))
/tmp/ipykernel_106/3705621695.py:2: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-x))
/tmp/ipykernel_106/3705621695.py:2: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-x))
/tmp/ipyke

Iteration 099 | Best Fitness: 0.44575 | Epoch Time: 5.85s
Training Complete
Total Time: 552.40 seconds
Average Epoch Time: 5.52 seconds
Number of selected features: 461


In [27]:
with open(output_path('selected_indices.csv'), 'w') as f:
    for idx in selected_indices:
        f.write(f"{idx}\n")

In [28]:
import pandas as pd

selected_features = pd.read_csv(output_path("selected_indices.csv"), header=None)[0].values

X_train = pd.read_csv(resolve_data_path("X_train.csv"))
X_val = pd.read_csv(resolve_data_path("X_val.csv"))
X_test = pd.read_csv(resolve_data_path("X_test.csv"))

y_train = pd.read_csv(resolve_data_path("y_train.csv")).iloc[:, 0].values
y_val = pd.read_csv(resolve_data_path("y_val.csv")).iloc[:, 0].values
y_test = pd.read_csv(resolve_data_path("y_test.csv")).iloc[:, 0].values

X_train_selected = X_train.iloc[:, selected_features].copy()
X_val_selected = X_val.iloc[:, selected_features].copy()
X_test_selected = X_test.iloc[:, selected_features].copy()

print("Train selected shape:", X_train_selected.shape)
print("Val selected shape:", X_val_selected.shape)
print("Test selected shape:", X_test_selected.shape)

Train selected shape: (16000, 461)
Val selected shape: (2000, 461)
Test selected shape: (2000, 461)


In [29]:
def compute_metrics(y_true, y_pred, model_name):
    report = classification_report(y_true, y_pred,
                                   target_names=CLASS_NAMES,
                                   output_dict=True, zero_division=0)
    row = {"model": model_name}
    for cls in CLASS_NAMES:
        for m in ["precision","recall","f1-score"]:
            col = f"{cls}_{m.replace('-score','')}"
            row[col] = round(report[cls][m], 4)
    for avg in ["macro avg","weighted avg"]:
        prefix = avg.replace(" avg","")
        for m in ["precision","recall","f1-score"]:
            col = f"{prefix}_{m.replace('-score','')}"
            row[col] = round(report[avg][m], 4)
    row["accuracy"] = round(report["accuracy"], 4)
    return row

In [30]:
all_results = [] 

from sklearn.neighbors         import KNeighborsClassifier
from sklearn.tree              import DecisionTreeClassifier
from sklearn.ensemble          import RandomForestClassifier
from sklearn.naive_bayes       import GaussianNB
from sklearn.linear_model      import LogisticRegression
from sklearn.svm               import LinearSVC, SVC
from sklearn.model_selection   import GridSearchCV
from sklearn.pipeline          import Pipeline
import numpy as np

unique_classes = np.unique(y_train)
N_CLASSES = len(unique_classes)
CLASS_NAMES = [str(c) for c in unique_classes]

print("N_CLASSES :", N_CLASSES)
print("CLASS_NAMES:", CLASS_NAMES)

ML_CONFIGS = [
    (
        "KNN",
        KNeighborsClassifier(),
        {"n_neighbors":[3,5,7,11], "weights":["uniform","distance"],
         "metric":["euclidean","manhattan"]}
    ),
    (
        "DecisionTree",
        DecisionTreeClassifier(random_state=42),
        {"max_depth":[None,10,20,30], "min_samples_split":[2,5,10],
         "criterion":["gini","entropy"]}
    ),
    (
        "RandomForest",
        RandomForestClassifier(random_state=42, n_jobs=-1),
        {"n_estimators":[100,200], "max_depth":[None,10,20],
         "min_samples_split":[2,5]}
    ),
    (
        "GaussianNB",
        GaussianNB(),
        {"var_smoothing":[1e-9,1e-8,1e-7,1e-6]}
    ),
    (
        "LogisticRegression",
        LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1),
        {"C":[0.01,0.1,1,10], "solver":["lbfgs","saga"],
         "multi_class":["multinomial"]}
    ),
    (
        "LinearSVM",
        LinearSVC(max_iter=2000, random_state=42),
        {"C":[0.01,0.1,1,10]}
    ),
    (
        "RBF_SVM",
        SVC(kernel="rbf", random_state=42),
        {"C":[0.1,1,10], "gamma":["scale","auto"]}
    ),
]

import pandas as pd
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report

all_results = []
report_rows = []

for name, estimator, param_grid in ML_CONFIGS:
    print(f"\n[{name}] GridSearch đang chạy...")

    gs = GridSearchCV(
        estimator,
        param_grid,
        cv=5,
        scoring="f1_macro",
        n_jobs=-1,
        verbose=0
    )

    gs.fit(X_train_selected, y_train)

    best = gs.best_estimator_
    y_pred = best.predict(X_test_selected)

    print(f"  Best params : {gs.best_params_}")
    print(f"  CV F1-macro : {gs.best_score_:.4f}")

    # report dạng dict thay vì in text thô
    report_dict = classification_report(
        y_test,
        y_pred,
        target_names=CLASS_NAMES,
        zero_division=0,
        output_dict=True
    )

    # Gom toàn bộ report của model thành 1 hàng
    flat_row = {"model": name}

    for label, metrics in report_dict.items():
        if isinstance(metrics, dict):  # các class, macro avg, weighted avg
            for metric_name, value in metrics.items():
                flat_row[f"{label}_{metric_name}"] = value
        else:  # accuracy là scalar
            flat_row[label] = metrics

    flat_row["best_params"] = str(gs.best_params_)
    flat_row["cv_f1_macro"] = gs.best_score_
    report_rows.append(flat_row)

    # phần cũ của bạn nếu vẫn muốn giữ
    row = compute_metrics(y_test, y_pred, name)
    row["best_params"] = str(gs.best_params_)
    all_results.append(row)

# Bảng classification report: mỗi hàng là 1 model
classification_report_df = pd.DataFrame(report_rows)

# Sắp xếp cột cho dễ nhìn
first_cols = ["model", "accuracy", "cv_f1_macro", "best_params"]
other_cols = [c for c in classification_report_df.columns if c not in first_cols]
classification_report_df = classification_report_df[first_cols + other_cols]

display(classification_report_df)

pd.DataFrame(all_results).to_csv(
    output_path("all_results_models.csv"),
    index=False,
    encoding="utf-8-sig"
)

N_CLASSES : 8
CLASS_NAMES: ['0', '1', '2', '3', '4', '5', '6', '7']

[KNN] GridSearch đang chạy...
  Best params : {'metric': 'manhattan', 'n_neighbors': 11, 'weights': 'distance'}
  CV F1-macro : 0.4919

[DecisionTree] GridSearch đang chạy...
  Best params : {'criterion': 'gini', 'max_depth': 10, 'min_samples_split': 2}
  CV F1-macro : 0.5650

[RandomForest] GridSearch đang chạy...
  Best params : {'max_depth': None, 'min_samples_split': 5, 'n_estimators': 200}
  CV F1-macro : 0.7069

[GaussianNB] GridSearch đang chạy...
  Best params : {'var_smoothing': 1e-06}
  CV F1-macro : 0.3477

[LogisticRegression] GridSearch đang chạy...


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and wi

  Best params : {'C': 0.01, 'multi_class': 'multinomial', 'solver': 'lbfgs'}
  CV F1-macro : 0.7081

[LinearSVM] GridSearch đang chạy...
  Best params : {'C': 0.01}
  CV F1-macro : 0.7038

[RBF_SVM] GridSearch đang chạy...
  Best params : {'C': 10, 'gamma': 'auto'}
  CV F1-macro : 0.7801


,model,accuracy,cv_f1_macro,best_params,0_precision,0_recall,0_f1-score,0_support,1_precision,1_recall,...,7_f1-score,7_support,macro avg_precision,macro avg_recall,macro avg_f1-score,macro avg_support,weighted avg_precision,weighted avg_recall,weighted avg_f1-score,weighted avg_support
0,KNN,0.5065,0.491907,"{'metric': 'manhattan', 'n_neighbors': 11, 'we...",0.516854,0.184,0.271386,250.0,0.319218,0.784,...,0.625000,250.0,0.575910,0.5065,0.506786,2000.0,0.575910,0.5065,0.506786,2000.0
1,DecisionTree,0.5825,0.564984,"{'criterion': 'gini', 'max_depth': 10, 'min_sa...",0.433862,0.328,0.373576,250.0,0.576792,0.676,...,0.627151,250.0,0.583517,0.5825,0.580055,2000.0,0.583517,0.5825,0.580055,2000.0
2,RandomForest,0.7330,0.706880,"{'max_depth': None, 'min_samples_split': 5, 'n...",0.668831,0.412,0.509901,250.0,0.682274,0.816,...,0.789588,250.0,0.741325,0.7330,0.729089,2000.0,0.741325,0.7330,0.729089,2000.0
3,GaussianNB,0.3670,0.347698,{'var_smoothing': 1e-06},0.109589,0.032,0.049536,250.0,0.364821,0.896,...,0.483871,250.0,0.397819,0.3670,0.347164,2000.0,0.397819,0.3670,0.347164,2000.0
4,LogisticRegression,0.7130,0.708125,"{'C': 0.01, 'multi_class': 'multinomial', 'sol...",0.608040,0.484,0.538976,250.0,0.701987,0.848,...,0.758197,250.0,0.710944,0.7130,0.709974,2000.0,0.710944,0.7130,0.709974,2000.0
5,LinearSVM,0.7075,0.703760,{'C': 0.01},0.638710,0.396,0.488889,250.0,0.699029,0.864,...,0.737903,250.0,0.704128,0.7075,0.700694,2000.0,0.704128,0.7075,0.700694,2000.0
6,RBF_SVM,0.7890,0.780109,"{'C': 10, 'gamma': 'auto'}",0.673820,0.628,0.650104,250.0,0.744681,0.840,...,0.843813,250.0,0.790561,0.7890,0.788959,2000.0,0.790561,0.7890,0.788959,2000.0
